In [16]:
from lsst.daf.butler import Butler
from lsst.pipe.tasks.calibrateImage import CalibrateImageTask
from lsst.ip.isr import IsrTask
import pandas as pd
import numpy as np

In [17]:
def isr (butler, dataId):
    raw = butler.get("raw", dataId=dataId)
    linearizer = butler.get("linearizer", dataId=dataId)
    ptc = butler.get("ptc", dataId=dataId)
    dark = butler.get("dark", dataId=dataId)
    bias = butler.get("bias", dataId=dataId)
    crosstalk = butler.get("crosstalk", dataId=dataId)
    defects = butler.get("defects", dataId=dataId)
    flat = butler.get("flat", dataId=dataId)
    cti = butler.get("cti", dataId=dataId)
    camera = butler.get("camera", dataId=dataId)

    cfg = IsrTask.ConfigClass()
    task = IsrTask(config=cfg)
    res = task.run(raw,
                   camera=camera,
                   bias=bias,
                   dark=dark,
                   flat=flat,
                   ptc=ptc,
                   linearizer=linearizer,
                   crosstalk=crosstalk,
                   defects=defects)
    return res.exposure

In [24]:
cfg = CalibrateImageTask.ConfigClass()
dir (cfg)

['ConnectionsClass',
 'ConnectionsConfigClass',
 '__annotations__',
 '__class__',
 '__contains__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_collectImports',
 '_fields',
 '_fromPython',
 '_frozen',
 '_history',
 '_imports',
 '_name',
 '_rename',
 '_save',
 '_source',
 '_storage',
 'applyConfigOverrides',
 'astrometry',
 'astrometry_ref_loader',
 'compare',
 'compute_summary_stats',
 'connections',
 'do_calibrate_pixels',
 'do_illumination_correction',
 'formatHistory',
 'freeze',
 'history',
 'id_generator',
 'install_simple_psf',
 'items',
 'keys',
 'load',
 'loadFromStream',
 'loadFromString',
 'measure_aperture_correction',
 'names',
 'option

In [45]:
def characterizeCalibrate(postISRCCD, threshold=5.0):

    cfg = CalibrateImageTask.ConfigClass()
    # disable astrometry
    #cfg.astrometry = None

    # disable photometry
    #cfg.photometry = None

    # detection threshold
    #cfg.star_detection.thresholdValue = threshold
    task = CalibrateImageTask(config=cfg)
    res = task.run(postisrccd)
    sourceCat = res.sourceCat
    return res.outputExposure, res.sourceCat

In [42]:
from lsst.pipe.tasks.calibrateImage import CalibrateImageTask

def characterizeCalibrate(postISRCCD, threshold=5.0):

    cfg = CalibrateImageTask.ConfigClass()

    # disable reference catalog usage
    #cfg.connections.astrometry_ref_cat = None
    #cfg.connections.photometry_ref_cat = None

    # detection threshold
    #cfg.star_detection.thresholdType = "stdev"
    #cfg.star_detection.thresholdValue = threshold

    task = CalibrateImageTask(config=cfg)
    res = task.run(postISRCCD)

    return res.exposure, res.sourceCat

In [43]:
repo = "dp2_prep"
collections = "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2"

butler = Butler(repo, collections=collections)

In [46]:
#postisrccd = isr (butler, {"instrument": 'LSSTCam', "detector": 1, "exposure": 2025091900155, "band": 'i'})
exposure, catalog = characterizeCalibrate(postisrccd, 5.0)

TypeError: CalibrateImageTask.run() takes 1 positional argument but 2 were given

In [12]:
catalog_stack = butler.get("single_visit_star_unstandardized", dataId={"instrument": 'LSSTCam', "detector": 1, "visit": 2025091900155, "band": 'i'})

In [13]:
len (catalog), len(catalog_stack)

(1126, 640)

In [ ]:
from lsst.pipe.tasks.calibrateImage import CalibrateImageTask

cfg = CalibrateImageTask.ConfigClass()
task = CalibrateImageTask(config=cfg)

res = task.run(postisrccd)

sourceCat = res.sourceCat